In [ ]:
from pathlib import Path
from pixelator import read_pna as read
from pixelator.pna.plot import molecule_rank_plot

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style("whitegrid")
import pandas as pd
from pixelator.common.statistics import clr_transformation, dsb_normalize
import scanpy as sc
from pixelator.mpx.plot import density_scatter_plot
import scanpy.external as sce
import networkx as nx

from pixelator.pna.analysis import calculate_differential_proximity


In [ ]:
ADATA_PATH='/home/projects/nyosef/zvise/PixelGen/PixelGen/adata_umap_latent.h5ad'
adata=sc.read(ADATA_PATH)

In [ ]:
sc.pl.umap(adata, color=['leiden','cell_group','condition'])

In [ ]:
sc.tl.rank_genes_groups(
    adata, "leiden", method="wilcoxon", layer="shifted_dsb"
)
diff_exp_df = sc.get.rank_genes_groups_df(adata, group=None)
diff_exp_df["-log10(adjusted p-value)"] = -np.log10(diff_exp_df["pvals_adj"])
diff_exp_df["Significant"] = diff_exp_df["pvals_adj"] < 0.01
diff_exp_df.head()

In [ ]:
def get_top_biologically_meaningful(diff_exp_df, n_top=20, known_markers=None):
    """
    Select top genes by significance + biological relevance.
    
    Scoring: significance (40%) + effect size (40%) + known marker bonus (20%)
    """
    if known_markers is None:
        known_markers = all_known_markers
    
    df = diff_exp_df.copy()
    df['nlog10_pval'] = -np.log10(df['pvals_adj'] + 1e-300)
    df['is_known_marker'] = df['names'].isin(known_markers)
    
    # Normalize scores to 0-1 range
    df['norm_pval'] = (df['nlog10_pval'] - df['nlog10_pval'].min()) / (df['nlog10_pval'].max() - df['nlog10_pval'].min() + 1e-10)
    df['norm_fc'] = (np.abs(df['logfoldchanges']) - np.abs(df['logfoldchanges']).min()) / (np.abs(df['logfoldchanges']).max() - np.abs(df['logfoldchanges']).min() + 1e-10)
    
    df['bio_score'] = (
        df['norm_pval'] * 0.4 +  # significance
        df['norm_fc'] * 0.4 +    # effect size
        df['is_known_marker'].astype(float) * 0.2  # bonus for known markers
    )
    
    # Get top genes ensuring mix of up and down regulated
    top_up = df[df['logfoldchanges'] > 0].nlargest(n_top // 2, 'bio_score')
    top_down = df[df['logfoldchanges'] < 0].nlargest(n_top // 2, 'bio_score')
    top_genes = pd.concat([top_up, top_down])['names'].tolist()
    
    return top_genes[:n_top]


def plot_heatmap_top_markers(adata, diff_exp_df, n_top=20, groupby='leiden'):
    """
    Plot heatmap of top biologically meaningful markers with row normalization.
    """
    top_genes = get_top_biologically_meaningful(diff_exp_df, n_top=n_top)
    
    # Filter to genes that exist in adata
    top_genes = [g for g in top_genes if g in adata.var_names]
    
    if len(top_genes) == 0:
        print("No matching genes found in adata")
        return
    
    print(f"Plotting top {len(top_genes)} biologically meaningful markers:")
    print(f"  Known immune markers: {[g for g in top_genes if g in all_known_markers]}")
    
    # Use scanpy heatmap with row normalization (standard_scale='var')
    sc.pl.heatmap(
        adata,
        var_names=top_genes,
        groupby=groupby,
        standard_scale='var',  # Row normalization (z-score per gene)
        cmap='RdBu_r',
        vmin=-2, vmax=2,
        figsize=(12, len(top_genes) * 0.4 + 2),
        swap_axes=True,
        show_gene_labels=True,
        dendrogram=True
    )


# Original heatmap code (kept for reference)
df = diff_exp_df.pivot(index=["names"], columns=["group"], values=["logfoldchanges"])

markers_for_heatmap = set(
    diff_exp_df[
        (np.abs(diff_exp_df["logfoldchanges"]) > 3) & diff_exp_df["Significant"]
    ]["names"]
)
markers_to_add = []
markers_for_heatmap.update(markers_to_add)
df = df[df.index.isin(markers_for_heatmap)]
df.columns = [cluster for _, cluster in df.columns]

# Create figure with row-normalized heatmap
from sklearn.preprocessing import StandardScaler

# Row normalize the data
df_normalized = pd.DataFrame(
    StandardScaler().fit_transform(df.T).T,  # z-score per row
    index=df.index,
    columns=df.columns
)

fig = sns.clustermap(
    df_normalized, 
    yticklabels=True, 
    linewidths=0.1, 
    cmap="RdBu_r",  # Changed to RdBu_r for better visibility
    vmin=-2, vmax=2,  # Standard z-score limits
    figsize=(10, 15)
)
fig.fig.suptitle('Top DE Markers (Row Normalized)', y=1.02, fontsize=14, fontweight='bold')

In [ ]:
sc.pl.umap(adata, color=['leiden','cell_group','condition'])

In [ ]:
# Known PBMC surface markers for biological relevance scoring
pbmc_surface_markers = {
    "T cells": ["CD3e", "CD2", "CD5", "CD7", "TCRab"],
    "CD4 T": ["CD4", "CD45RA", "CD45RO", "CD27", "CD28"],
    "CD8 T": ["CD8", "CD57", "KLRG1", "CX3CR1"],
    "Gamma-delta T": ["TCRgd", "TCRVg9", "TCRVd2"],
    "NK cells": ["CD56", "CD94", "CD335", "NKp80", "CD314"],
    "B cells": ["CD19", "CD20", "CD22", "CD79a", "CD37"],
    "Plasma cells": ["CD138", "CD38", "CD319"],
    "Monocytes": ["CD14", "CD16", "CD11b", "CD33", "CD64"],
    "DCs": ["CD11c", "HLA-DR", "CD86", "CD80"],
    "Platelets": ["CD41", "CD62P", "CD9", "CD36"],
}
all_known_markers = set(m for markers in pbmc_surface_markers.values() for m in markers)


def calculate_pct_expressing(adata, genes, groupby, layer='shifted_dsb', threshold=0):
    """Calculate percent of cells expressing each gene per group."""
    pct_results = {}
    for gene in genes:
        if gene not in adata.var_names:
            continue
        pct_dict = {}
        gene_idx = list(adata.var_names).index(gene)
        for group in adata.obs[groupby].unique():
            mask = adata.obs[groupby] == group
            if layer and layer in adata.layers:
                values = adata[mask].layers[layer][:, gene_idx]
            else:
                values = adata[mask].X[:, gene_idx]
            values = values.toarray().flatten() if hasattr(values, 'toarray') else np.array(values).flatten()
            pct_dict[group] = (values > threshold).mean() * 100
        pct_results[gene] = pct_dict
    return pct_results


def DE_by_group(adata, condition, reference, groups, cell_group=None, cell_type=None, 
                plot_volcano=True, use_pct_size=True, fc_thresh=1.5, n_labels=15):
    """
    Differential expression analysis with enhanced volcano plot.
    
    Parameters:
    -----------
    use_pct_size : bool
        If True, dot size is proportional to percent of expressing cells
    fc_thresh : float
        Fold change threshold for labeling genes
    n_labels : int
        Maximum number of genes to label
    """
    if cell_group is not None:
        adata = adata[adata.obs['cell_group'] == cell_group].copy()
    if cell_type is not None:
        adata = adata[adata.obs['cell_type'] == cell_type].copy()
    
    print(f"Analyzing {adata.obs.shape[0]} cells")
    sc.tl.rank_genes_groups(adata, groupby=condition, method='wilcoxon', 
                            groups=groups, reference=reference, layer='shifted_dsb')
    diff_exp_df = sc.get.rank_genes_groups_df(adata, group=None)
    diff_exp_df["-log10(adjusted p-value)"] = -np.log10(diff_exp_df["pvals_adj"])
    diff_exp_df["Significant"] = diff_exp_df["pvals_adj"] < 0.01
    
    # Calculate percent expressing for the test group
    if use_pct_size:
        pct_results = calculate_pct_expressing(adata, diff_exp_df['names'].unique(), 
                                               condition, layer='shifted_dsb')
        test_group = groups[0]
        diff_exp_df['pct_expressing'] = diff_exp_df['names'].apply(
            lambda x: pct_results.get(x, {}).get(test_group, 0)
        )
    
    if plot_volcano:
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Color palette for significance direction
        diff_exp_df['Direction'] = np.where(
            ~diff_exp_df['Significant'], 'NS',
            np.where(diff_exp_df['logfoldchanges'] > 0, 'Up', 'Down')
        )
        palette = {'Up': '#d62728', 'Down': '#1f77b4', 'NS': 'lightgrey'}
        
        if use_pct_size:
            # Dot size proportional to percent expressing
            scatter = sns.scatterplot(
                data=diff_exp_df, 
                x='logfoldchanges', 
                y='-log10(adjusted p-value)', 
                hue='Direction',
                size='pct_expressing',
                sizes=(20, 300),
                palette=palette,
                alpha=0.7,
                ax=ax
            )
            # Move size legend to better position
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(handles, labels, loc='upper right', title='Direction / % Expressing')
        else:
            sns.scatterplot(
                data=diff_exp_df, 
                x='logfoldchanges', 
                y='-log10(adjusted p-value)', 
                hue='Direction',
                palette=palette,
                alpha=0.7,
                ax=ax
            )
        
        # Add threshold lines
        ax.axhline(-np.log10(0.01), color='grey', linestyle='--', linewidth=1, alpha=0.5)
        ax.axvline(fc_thresh, color='grey', linestyle='--', linewidth=1, alpha=0.5)
        ax.axvline(-fc_thresh, color='grey', linestyle='--', linewidth=1, alpha=0.5)
        
        # Label top genes
        sig_df = diff_exp_df[diff_exp_df['Significant']].copy()
        top_up = sig_df[sig_df['logfoldchanges'] > fc_thresh].nlargest(n_labels // 2, 'logfoldchanges')
        top_down = sig_df[sig_df['logfoldchanges'] < -fc_thresh].nsmallest(n_labels // 2, 'logfoldchanges')
        to_label = pd.concat([top_up, top_down])
        
        for _, row in to_label.iterrows():
            ax.annotate(
                row['names'], 
                (row['logfoldchanges'], row['-log10(adjusted p-value)']),
                xytext=(5 if row['logfoldchanges'] > 0 else -5, 5), 
                textcoords='offset points',
                fontsize=9,
                ha='left' if row['logfoldchanges'] > 0 else 'right'
            )
        
        title = f'DE: {reference} vs {groups[0]}'
        if cell_group:
            title += f' (cell group: {cell_group})'
        if cell_type:
            title += f' (cell type: {cell_type})'
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_xlabel('Log2 Fold Change', fontsize=12)
        ax.set_ylabel('-log10(adjusted p-value)', fontsize=12)
        
        plt.tight_layout()
        plt.show()
    
    return diff_exp_df

In [ ]:
diff_exp_df=DE_by_group(adata=adata, condition='leiden',reference='8',groups=['2'])
display(diff_exp_df.head(10))

In [ ]:
display(diff_exp_df.tail(10))

# Task 2: Top 20 Biologically Meaningful Heatmap

Use the enhanced function to plot top markers with row normalization.

In [ ]:
# Plot top 20 biologically meaningful markers with row normalization
plot_heatmap_top_markers(adata, diff_exp_df, n_top=20, groupby='leiden')

# Task 3: Pseudobulk Differential Expression Analysis

Perform pseudobulk DE analysis for:
- **Abundance**: Protein expression levels
- **Hotspot**: Spatial colocalization features (Moran's I based)
- **Joint count**: Proximity-based colocalization features

Compare PHA vs unstimulated samples.

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA

# ============================================================================
# PSEUDOBULK DIFFERENTIAL EXPRESSION ANALYSIS
# ============================================================================

def create_pseudobulk(adata, groupby='sample', layer=None, obsm_key=None):
    """
    Aggregate cells per sample to create pseudobulk profiles.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data matrix
    groupby : str
        Column in adata.obs to group by (e.g., 'sample', 'sample_id')
    layer : str, optional
        Layer to use for abundance data
    obsm_key : str, optional
        Key in adata.obsm for spatial features
    
    Returns:
    --------
    pseudobulk : DataFrame
        Pseudobulk expression matrix (samples x features)
    metadata : DataFrame
        Sample metadata including condition
    """
    if obsm_key and obsm_key in adata.obsm:
        data = pd.DataFrame(adata.obsm[obsm_key], index=adata.obs_names)
    elif layer and layer in adata.layers:
        data = pd.DataFrame(
            adata.layers[layer].toarray() if hasattr(adata.layers[layer], 'toarray') else adata.layers[layer],
            index=adata.obs_names, 
            columns=adata.var_names
        )
    else:
        X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
        data = pd.DataFrame(X, index=adata.obs_names, columns=adata.var_names)
    
    data['_group'] = adata.obs[groupby].values
    
    # Aggregate by mean
    pseudobulk = data.groupby('_group').mean()
    
    # Get metadata for each sample
    meta_cols = ['condition'] if 'condition' in adata.obs.columns else []
    if 'cell_group' in adata.obs.columns:
        meta_cols.append('cell_group')
    
    metadata = adata.obs.groupby(groupby)[meta_cols].first() if meta_cols else pd.DataFrame(index=pseudobulk.index)
    metadata['n_cells'] = adata.obs.groupby(groupby).size()
    
    return pseudobulk, metadata


def run_pseudobulk_de(pseudobulk, metadata, test_condition='PHA', ref_condition='unstim', 
                      condition_col='condition'):
    """
    Run t-test based DE on pseudobulk data.
    
    Returns DataFrame with logFC, pvalue, pval_adj, mean values
    """
    if condition_col not in metadata.columns:
        raise ValueError(f"Condition column '{condition_col}' not found in metadata")
    
    test_samples = metadata[metadata[condition_col] == test_condition].index
    ref_samples = metadata[metadata[condition_col] == ref_condition].index
    
    if len(test_samples) < 2 or len(ref_samples) < 2:
        raise ValueError(f"Need at least 2 samples per condition. Got {len(test_samples)} test, {len(ref_samples)} ref")
    
    results = []
    for feat in pseudobulk.columns:
        test_vals = pseudobulk.loc[test_samples, feat].values
        ref_vals = pseudobulk.loc[ref_samples, feat].values
        
        # Handle constant values
        if np.std(test_vals) == 0 and np.std(ref_vals) == 0:
            t_stat, pval = 0, 1
        else:
            t_stat, pval = stats.ttest_ind(test_vals, ref_vals, equal_var=False)
        
        # Log fold change (add small constant to avoid division by zero)
        mean_test = np.mean(test_vals)
        mean_ref = np.mean(ref_vals)
        log_fc = np.log2((mean_test + 1e-6) / (mean_ref + 1e-6))
        
        results.append({
            'feature': feat,
            'logFC': log_fc,
            'pvalue': pval if not np.isnan(pval) else 1,
            't_stat': t_stat if not np.isnan(t_stat) else 0,
            f'mean_{test_condition}': mean_test,
            f'mean_{ref_condition}': mean_ref
        })
    
    results_df = pd.DataFrame(results)
    
    # Multiple testing correction
    _, pvals_adj, _, _ = multipletests(results_df['pvalue'], method='fdr_bh')
    results_df['pval_adj'] = pvals_adj
    results_df['nlog10_pval'] = -np.log10(results_df['pval_adj'] + 1e-300)
    
    # Classify significance
    results_df['significant'] = results_df['pval_adj'] < 0.05
    results_df['direction'] = np.where(
        ~results_df['significant'], 'NS',
        np.where(results_df['logFC'] > 0, f'Up in {test_condition}', f'Up in {ref_condition}')
    )
    
    return results_df.sort_values('pval_adj')


def plot_pseudobulk_volcano(results_df, title, fc_thresh=0.5, pval_thresh=0.05, 
                            n_labels=10, figsize=(10, 8)):
    """Plot volcano plot for pseudobulk DE results."""
    fig, ax = plt.subplots(figsize=figsize)
    
    # Build palette: NS=grey, positive logFC direction=red, negative=blue
    directions = results_df['direction'].unique()
    palette = {}
    for d in directions:
        if d == 'NS':
            palette[d] = 'lightgrey'
        else:
            # Determine color by checking if this direction has positive or negative logFC
            dir_mask = results_df['direction'] == d
            if dir_mask.any() and results_df.loc[dir_mask, 'logFC'].mean() > 0:
                palette[d] = '#d62728'  # Red for upregulated in test
            else:
                palette[d] = '#1f77b4'  # Blue for upregulated in reference
    
    sns.scatterplot(
        data=results_df,
        x='logFC',
        y='nlog10_pval',
        hue='direction',
        palette=palette,
        alpha=0.7,
        s=50,
        ax=ax
    )
    
    # Threshold lines
    ax.axhline(-np.log10(pval_thresh), color='grey', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(fc_thresh, color='grey', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(-fc_thresh, color='grey', linestyle='--', linewidth=1, alpha=0.5)
    
    # Label top hits
    sig_df = results_df[results_df['significant']].copy()
    top_up = sig_df[sig_df['logFC'] > fc_thresh].nlargest(n_labels // 2, 'logFC')
    top_down = sig_df[sig_df['logFC'] < -fc_thresh].nsmallest(n_labels // 2, 'logFC')
    to_label = pd.concat([top_up, top_down])
    
    for _, row in to_label.iterrows():
        ax.annotate(
            row['feature'][:20],  # Truncate long feature names
            (row['logFC'], row['nlog10_pval']),
            xytext=(5 if row['logFC'] > 0 else -5, 5),
            textcoords='offset points',
            fontsize=8,
            ha='left' if row['logFC'] > 0 else 'right'
        )
    
    ax.set_xlabel('Log2 Fold Change', fontsize=12)
    ax.set_ylabel('-log10(adjusted p-value)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(title='Direction', loc='upper right')
    
    plt.tight_layout()
    return fig, ax


def plot_pseudobulk_ma(results_df, title, figsize=(10, 6)):
    """MA plot: log mean expression vs log fold change."""
    fig, ax = plt.subplots(figsize=figsize)
    
    # Calculate A (average expression)
    results_df = results_df.copy()
    mean_cols = [c for c in results_df.columns if c.startswith('mean_')]
    results_df['A'] = results_df[mean_cols].mean(axis=1).apply(lambda x: np.log2(x + 1e-6))
    
    # Build palette
    palette = {}
    for d in results_df['direction'].unique():
        if d == 'NS':
            palette[d] = 'lightgrey'
        else:
            dir_mask = results_df['direction'] == d
            if dir_mask.any() and results_df.loc[dir_mask, 'logFC'].mean() > 0:
                palette[d] = '#d62728'
            else:
                palette[d] = '#1f77b4'
    
    sns.scatterplot(
        data=results_df,
        x='A',
        y='logFC',
        hue='direction',
        palette=palette,
        alpha=0.6,
        s=30,
        ax=ax
    )
    
    ax.axhline(0, color='grey', linestyle='-', linewidth=1, alpha=0.5)
    ax.set_xlabel('Average Expression (log2)', fontsize=12)
    ax.set_ylabel('Log2 Fold Change', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    return fig, ax


def plot_pseudobulk_pca(pseudobulk, metadata, condition_col='condition', title='Pseudobulk PCA'):
    """PCA of pseudobulk samples colored by condition."""
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Run PCA
    pca = PCA(n_components=min(2, pseudobulk.shape[0], pseudobulk.shape[1]))
    pcs = pca.fit_transform(pseudobulk.values)
    
    if pcs.shape[1] < 2:
        print("Not enough components for 2D PCA")
        plt.close(fig)
        return fig, ax
    
    # Create plot dataframe
    plot_df = pd.DataFrame({
        'PC1': pcs[:, 0],
        'PC2': pcs[:, 1],
        'condition': metadata[condition_col].values,
        'sample': pseudobulk.index
    })
    
    sns.scatterplot(
        data=plot_df,
        x='PC1', y='PC2',
        hue='condition',
        s=100,
        ax=ax
    )
    
    # Label points
    for _, row in plot_df.iterrows():
        ax.annotate(row['sample'], (row['PC1'], row['PC2']), 
                   xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    return fig, ax


def plot_top_features_heatmap(pseudobulk, metadata, results_df, n_top=15, 
                              condition_col='condition', title='Top DE Features'):
    """Heatmap of top DE features."""
    # Get top features
    sig = results_df[results_df['significant']].copy()
    if len(sig) == 0:
        print("No significant features to plot")
        return
    
    top_up = sig[sig['logFC'] > 0].nlargest(n_top // 2, 'nlog10_pval')
    top_down = sig[sig['logFC'] < 0].nlargest(n_top // 2, 'nlog10_pval')
    top_feats = pd.concat([top_up, top_down])['feature'].tolist()
    
    if len(top_feats) == 0:
        print("No features to plot")
        return
    
    # Filter to features that exist in pseudobulk
    top_feats = [f for f in top_feats if f in pseudobulk.columns]
    if len(top_feats) == 0:
        print("No matching features in pseudobulk")
        return
    
    # Subset and normalize
    plot_data = pseudobulk[top_feats].copy()
    
    # Z-score normalize per feature (row)
    from sklearn.preprocessing import StandardScaler
    plot_data_norm = pd.DataFrame(
        StandardScaler().fit_transform(plot_data),
        index=plot_data.index,
        columns=plot_data.columns
    )
    
    # Build condition color map dynamically
    unique_conditions = metadata[condition_col].unique()
    default_colors = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
    cond_color_map = {c: default_colors[i % len(default_colors)] for i, c in enumerate(unique_conditions)}
    row_colors = metadata[condition_col].map(cond_color_map)
    
    g = sns.clustermap(
        plot_data_norm.T,
        col_colors=row_colors,
        cmap='RdBu_r',
        vmin=-2, vmax=2,
        figsize=(10, max(8, len(top_feats) * 0.3)),
        yticklabels=True,
        xticklabels=True
    )
    g.fig.suptitle(title, y=1.02, fontsize=14, fontweight='bold')
    
    return g


print("Pseudobulk DE functions defined successfully!")

In [ ]:
# Check available data for pseudobulk analysis
print("Available layers:", list(adata.layers.keys()) if adata.layers else "None")
print("Available obsm keys:", list(adata.obsm.keys()) if adata.obsm else "None")
print("\nObs columns:", list(adata.obs.columns))

# Check for sample/condition columns
sample_cols = [c for c in adata.obs.columns if 'sample' in c.lower()]
print(f"\nPossible sample columns: {sample_cols}")
print(f"Condition column exists: {'condition' in adata.obs.columns}")

if 'condition' in adata.obs.columns:
    print(f"\nCondition values: {adata.obs['condition'].value_counts().to_dict()}")

In [ ]:
# ============================================================================
# RUN PSEUDOBULK DE ANALYSIS
# ============================================================================

# Determine groupby column (sample identifier)
if 'sample_id' in adata.obs.columns:
    groupby = 'sample_id'
elif 'sample' in adata.obs.columns:
    groupby = 'sample'
else:
    # Create a sample column from condition if not available
    # This is a fallback - in real data you'd have true sample IDs
    print("Warning: No sample column found. Creating pseudo-samples from condition.")
    adata.obs['pseudo_sample'] = adata.obs['condition'].astype(str) + '_' + \
                                  (adata.obs.groupby('condition').cumcount() // 500).astype(str)
    groupby = 'pseudo_sample'

print(f"Using '{groupby}' for pseudobulk aggregation")
print(f"Number of groups: {adata.obs[groupby].nunique()}")

# Determine condition values
if 'condition' in adata.obs.columns:
    conditions = adata.obs['condition'].unique()
    print(f"Conditions: {conditions}")
    
    # Try to identify test vs reference
    test_cond = [c for c in conditions if 'pha' in str(c).lower() or 'cart' in str(c).lower() or 'stim' in str(c).lower()]
    ref_cond = [c for c in conditions if 'unstim' in str(c).lower() or 'control' in str(c).lower()]
    
    if test_cond and ref_cond:
        TEST_CONDITION = test_cond[0]
        REF_CONDITION = ref_cond[0]
    else:
        TEST_CONDITION = conditions[0]
        REF_CONDITION = conditions[1] if len(conditions) > 1 else conditions[0]
    
    print(f"Test condition: {TEST_CONDITION}")
    print(f"Reference condition: {REF_CONDITION}")

In [ ]:
# ============================================================================
# ABUNDANCE PSEUDOBULK DE
# ============================================================================

# Define modalities to analyze
modalities = {}

# Abundance layer
if 'shifted_dsb' in adata.layers:
    modalities['Abundance'] = {'layer': 'shifted_dsb'}
elif 'clr' in adata.layers:
    modalities['Abundance'] = {'layer': 'clr'}
else:
    modalities['Abundance'] = {'layer': None}  # Use X

# Spatial features (if available)
if 'HOTSPOT_top500_var' in adata.obsm:
    modalities['Hotspot'] = {'obsm_key': 'HOTSPOT_top500_var'}
if 'spatial_asinh5_top500_var' in adata.obsm:
    modalities['JoinCount'] = {'obsm_key': 'spatial_asinh5_top500_var'}
if 'HOTSPOT' in adata.obsm:
    modalities['Hotspot'] = {'obsm_key': 'HOTSPOT'}

print(f"Modalities to analyze: {list(modalities.keys())}")

# Store results
all_results = {}

for mod_name, params in modalities.items():
    print(f"\n{'='*60}")
    print(f"Analyzing: {mod_name}")
    print('='*60)
    
    try:
        # Create pseudobulk
        pseudobulk, metadata = create_pseudobulk(adata, groupby=groupby, **params)
        print(f"Pseudobulk shape: {pseudobulk.shape}")
        print(f"Samples per condition: {metadata['condition'].value_counts().to_dict()}")
        
        # Check if we have enough samples
        if metadata['condition'].value_counts().min() < 2:
            print(f"  Skipping {mod_name}: Need at least 2 samples per condition")
            continue
        
        # Run DE
        de_results = run_pseudobulk_de(
            pseudobulk, metadata, 
            test_condition=TEST_CONDITION, 
            ref_condition=REF_CONDITION
        )
        all_results[mod_name] = {
            'pseudobulk': pseudobulk,
            'metadata': metadata,
            'de_results': de_results
        }
        
        # Summary
        n_sig = de_results['significant'].sum()
        n_up = (de_results['direction'].str.contains('Up in ' + TEST_CONDITION, na=False)).sum()
        n_down = (de_results['direction'].str.contains('Up in ' + REF_CONDITION, na=False)).sum()
        print(f"\nResults: {n_sig} significant features ({n_up} up, {n_down} down)")
        
        # Display top results
        print(f"\nTop 5 features (by adjusted p-value):")
        display(de_results.head(5)[['feature', 'logFC', 'pval_adj', 'direction']])
        
    except Exception as e:
        print(f"  Error analyzing {mod_name}: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# ============================================================================
# PSEUDOBULK VISUALIZATIONS
# ============================================================================

for mod_name, data in all_results.items():
    print(f"\n{'='*60}")
    print(f"Visualizations for: {mod_name}")
    print('='*60)
    
    de_results = data['de_results']
    pseudobulk = data['pseudobulk']
    metadata = data['metadata']
    
    # 1. Volcano Plot
    fig, ax = plot_pseudobulk_volcano(
        de_results, 
        title=f'{mod_name}: Pseudobulk DE ({TEST_CONDITION} vs {REF_CONDITION})',
        fc_thresh=0.5,
        n_labels=12
    )
    plt.show()
    
    # 2. MA Plot
    fig, ax = plot_pseudobulk_ma(
        de_results,
        title=f'{mod_name}: MA Plot'
    )
    plt.show()
    
    # 3. PCA
    if len(pseudobulk) >= 3:  # Need at least 3 samples for meaningful PCA
        fig, ax = plot_pseudobulk_pca(
            pseudobulk, metadata,
            title=f'{mod_name}: Pseudobulk PCA'
        )
        plt.show()
    
    # 4. Heatmap of top features
    if de_results['significant'].sum() > 0:
        plot_top_features_heatmap(
            pseudobulk, metadata, de_results,
            n_top=20,
            title=f'{mod_name}: Top DE Features Heatmap'
        )
        plt.show()

In [ ]:
# ============================================================================
# RESULTS TABLES
# ============================================================================

for mod_name, data in all_results.items():
    de_results = data['de_results']
    sig_results = de_results[de_results['significant']].copy()
    
    print(f"\n{'='*60}")
    print(f"{mod_name}: Significant Features Table")
    print('='*60)
    print(f"Total significant: {len(sig_results)}")
    
    if len(sig_results) > 0:
        # Format for display
        display_cols = ['feature', 'logFC', 'pval_adj', 'direction']
        if f'mean_{TEST_CONDITION}' in sig_results.columns:
            display_cols.extend([f'mean_{TEST_CONDITION}', f'mean_{REF_CONDITION}'])
        
        display(sig_results[display_cols].head(30).style.format({
            'logFC': '{:.3f}',
            'pval_adj': '{:.2e}',
            f'mean_{TEST_CONDITION}': '{:.3f}' if f'mean_{TEST_CONDITION}' in display_cols else '{}',
            f'mean_{REF_CONDITION}': '{:.3f}' if f'mean_{REF_CONDITION}' in display_cols else '{}'
        }).background_gradient(subset=['logFC'], cmap='RdBu_r', vmin=-2, vmax=2))

# Task 4: Marker Community Detection

Find groups of markers that are close together (colocalized), not just pairs.

Uses graph-based community detection:
1. Build graph where nodes = markers, edges = high colocalization pairs
2. Apply Louvain/Leiden community detection to find marker modules
3. Visualize and interpret the marker communities

In [ ]:
# ============================================================================
# MARKER COMMUNITY DETECTION
# ============================================================================

try:
    from community import community_louvain
    HAS_LOUVAIN = True
except ImportError:
    print("python-louvain not installed. Install with: pip install python-louvain")
    HAS_LOUVAIN = False


def build_marker_colocalization_graph(coloc_df, threshold=0.2, use_abs=True):
    """
    Build a graph from colocalization data.
    
    Parameters:
    -----------
    coloc_df : DataFrame
        Colocalization data with columns like 'CD4/CD8' containing correlation values
    threshold : float
        Minimum correlation threshold to create an edge
    use_abs : bool
        If True, use absolute correlation values
    
    Returns:
    --------
    G : nx.Graph
        Network graph with markers as nodes and correlations as edge weights
    """
    G = nx.Graph()
    
    for col in coloc_df.columns:
        if '/' in col:
            parts = col.split('/')
            if len(parts) == 2:
                m1, m2 = parts
                if m1 != m2:  # Skip self-loops
                    # Calculate mean correlation across all cells
                    weight = coloc_df[col].mean()
                    if use_abs:
                        weight = abs(weight)
                    
                    if weight >= threshold:
                        # Add edge with weight
                        if G.has_edge(m1, m2):
                            # Average if edge exists
                            G[m1][m2]['weight'] = (G[m1][m2]['weight'] + weight) / 2
                        else:
                            G.add_edge(m1, m2, weight=weight)
    
    return G


def find_marker_communities(G, resolution=1.0):
    """
    Find marker communities using Louvain algorithm.
    
    Returns dict: {community_id: [list of markers]}
    """
    if not HAS_LOUVAIN:
        print("Louvain not available. Using connected components instead.")
        communities = {}
        for i, comp in enumerate(nx.connected_components(G)):
            communities[i] = list(comp)
        return communities, {}
    
    # Run Louvain community detection
    partition = community_louvain.best_partition(G, resolution=resolution, random_state=42)
    
    # Group markers by community
    communities = {}
    for marker, comm_id in partition.items():
        if comm_id not in communities:
            communities[comm_id] = []
        communities[comm_id].append(marker)
    
    return communities, partition


def visualize_marker_network(G, partition, title='Marker Colocalization Network', figsize=(14, 10)):
    """Visualize the marker network with community coloring."""
    if len(G.nodes()) == 0:
        print("No nodes in graph to visualize")
        return
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Layout
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    # Get colors for communities
    if partition:
        colors = [partition.get(node, 0) for node in G.nodes()]
        cmap = plt.cm.Set3
    else:
        colors = 'lightblue'
        cmap = None
    
    # Draw edges with width proportional to weight
    edges = G.edges()
    weights = [G[u][v]['weight'] * 3 for u, v in edges]
    
    nx.draw_networkx_edges(G, pos, alpha=0.3, width=weights, edge_color='gray', ax=ax)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=500, alpha=0.8, cmap=cmap, ax=ax)
    
    # Draw labels
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    
    plt.tight_layout()
    return fig, ax


def interpret_communities(communities, known_markers_dict):
    """
    Interpret communities by matching with known marker sets.
    """
    interpretations = {}
    
    for comm_id, markers in communities.items():
        marker_set = set(markers)
        matches = []
        
        for cell_type, known in known_markers_dict.items():
            overlap = marker_set.intersection(set(known))
            if len(overlap) > 0:
                matches.append((cell_type, overlap, len(overlap) / len(known)))
        
        # Sort by overlap fraction
        matches.sort(key=lambda x: -x[2])
        interpretations[comm_id] = {
            'markers': markers,
            'size': len(markers),
            'matches': matches[:3]  # Top 3 matches
        }
    
    return interpretations


print("Marker community detection functions defined!")

In [ ]:
# ============================================================================
# RUN MARKER COMMUNITY DETECTION
# ============================================================================

# Check for colocalization data
coloc_keys = [k for k in adata.obsm.keys() if 'hotspot' in k.lower() or 'coloc' in k.lower() or 'spatial' in k.lower()]
print(f"Available colocalization data: {coloc_keys}")

if len(coloc_keys) > 0:
    # Use the first available colocalization dataset
    coloc_key = coloc_keys[0]
    print(f"\nUsing: {coloc_key}")
    
    coloc_data = pd.DataFrame(adata.obsm[coloc_key])
    print(f"Shape: {coloc_data.shape}")
    
    # Check column format
    pair_cols = [c for c in coloc_data.columns if '/' in str(c)]
    print(f"Pair columns found: {len(pair_cols)}")
    
    if len(pair_cols) > 0:
        # Build the graph
        print("\nBuilding marker colocalization graph...")
        G = build_marker_colocalization_graph(coloc_data, threshold=0.15, use_abs=True)
        print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        
        # Find communities
        print("\nDetecting communities...")
        communities, partition = find_marker_communities(G, resolution=1.0)
        
        # Print results
        print(f"\n{'='*60}")
        print("MARKER COMMUNITIES FOUND")
        print('='*60)
        
        for comm_id, markers in sorted(communities.items(), key=lambda x: -len(x[1])):
            if len(markers) >= 2:  # Only show communities with 2+ markers
                print(f"\nCommunity {comm_id} ({len(markers)} markers):")
                print(f"  {', '.join(sorted(markers))}")
        
        # Interpret communities
        print(f"\n{'='*60}")
        print("BIOLOGICAL INTERPRETATION")
        print('='*60)
        
        interpretations = interpret_communities(communities, pbmc_surface_markers)
        for comm_id, info in sorted(interpretations.items(), key=lambda x: -x[1]['size']):
            if info['size'] >= 2:
                print(f"\nCommunity {comm_id} ({info['size']} markers):")
                if info['matches']:
                    for cell_type, overlap, frac in info['matches']:
                        print(f"  → {cell_type}: {sorted(overlap)} ({frac*100:.0f}% overlap)")
                else:
                    print(f"  → No clear cell type match")
    else:
        print("No pair columns (format: 'MarkerA/MarkerB') found in data")
else:
    print("No colocalization data found in adata.obsm")
    print("Available keys:", list(adata.obsm.keys()) if adata.obsm else "None")

In [ ]:
# ============================================================================
# MARKER COMMUNITY VISUALIZATIONS
# ============================================================================

if 'G' in dir() and G.number_of_nodes() > 0:
    # 1. Network visualization with community coloring
    fig, ax = visualize_marker_network(
        G, partition, 
        title='Marker Colocalization Network (Communities)',
        figsize=(14, 10)
    )
    plt.show()
    
    # 2. Heatmap of within-community correlations
    if len(pair_cols) > 0:
        # Build mean correlation matrix between markers
        unique_markers = sorted(G.nodes())
        n_markers = len(unique_markers)
        corr_matrix = pd.DataFrame(
            np.zeros((n_markers, n_markers)),
            index=unique_markers, columns=unique_markers
        )
        
        for col in pair_cols:
            parts = col.split('/')
            if len(parts) == 2:
                m1, m2 = parts
                if m1 in unique_markers and m2 in unique_markers:
                    val = coloc_data[col].mean()
                    corr_matrix.loc[m1, m2] = val
                    corr_matrix.loc[m2, m1] = val
        
        # Sort by community
        if partition:
            marker_order = sorted(unique_markers, key=lambda x: (partition.get(x, 999), x))
            corr_matrix = corr_matrix.loc[marker_order, marker_order]
            
            # Create color labels for communities
            comm_colors = pd.Series(
                [partition.get(m, 0) for m in marker_order],
                index=marker_order
            )
            lut = dict(zip(sorted(set(comm_colors)), sns.color_palette("Set3", len(set(comm_colors)))))
            row_colors = comm_colors.map(lut)
            
            g = sns.clustermap(
                corr_matrix,
                cmap='RdBu_r',
                vmin=-0.5, vmax=0.5,
                figsize=(12, 12),
                row_cluster=False,
                col_cluster=False,
                row_colors=row_colors,
                col_colors=row_colors,
                yticklabels=True,
                xticklabels=True
            )
            g.fig.suptitle('Marker Colocalization Matrix (sorted by community)', y=1.02, fontsize=14, fontweight='bold')
            plt.show()
    
    # 3. Per-condition community analysis 
    if 'condition' in adata.obs.columns:
        conditions = adata.obs['condition'].unique()
        
        fig, axes = plt.subplots(1, len(conditions), figsize=(7 * len(conditions), 6))
        if len(conditions) == 1:
            axes = [axes]
        
        for idx, cond in enumerate(conditions):
            ax = axes[idx]
            mask = adata.obs['condition'] == cond
            cond_coloc = coloc_data[mask]
            
            G_cond = build_marker_colocalization_graph(cond_coloc, threshold=0.15, use_abs=True)
            
            # Use same positions as main graph for comparability
            pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
            
            # Only plot nodes/edges present in this condition's graph
            common_nodes = set(G.nodes()).intersection(set(G_cond.nodes()))
            subpos = {n: pos[n] for n in common_nodes if n in pos}
            
            if len(common_nodes) > 0:
                edges = G_cond.edges()
                weights = [G_cond[u][v]['weight'] * 3 for u, v in edges]
                
                nx.draw_networkx_edges(G_cond, subpos, alpha=0.3, width=weights, edge_color='gray', ax=ax)
                colors = [partition.get(node, 0) for node in G_cond.nodes() if node in subpos]
                nodes = [n for n in G_cond.nodes() if n in subpos]
                nx.draw_networkx_nodes(G_cond, subpos, nodelist=nodes, node_color=colors, 
                                      node_size=400, alpha=0.8, cmap=plt.cm.Set3, ax=ax)
                nx.draw_networkx_labels(G_cond, subpos, font_size=7, ax=ax)
            
            ax.set_title(f'{cond}\n({G_cond.number_of_nodes()} nodes, {G_cond.number_of_edges()} edges)', fontsize=12)
            ax.axis('off')
        
        plt.suptitle('Marker Networks by Condition', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("No marker graph available. Skipping visualizations.")